# Time Series Analysis & Forecasting ⏰

Predicting the future using the past! Stock prices, weather, sales, website traffic—all time series data.

## What You'll Learn
- Time series components (trend, seasonality, noise)
- Autocorrelation and stationarity
- ARIMA models
- Exponential smoothing
- Forecasting evaluation metrics
- Real-world examples: stock prices, weather, sales

## Time Series Basics 📈

### What is Time Series Data?
Data points collected at regular intervals over time. The order matters!

**Examples:**
- Stock prices (daily)
- Temperature (hourly)
- Website traffic (per minute)
- Monthly sales
- Daily COVID cases

### Key Difference from Regular Data
- **Regular ML:** Features are independent
- **Time Series:** Data points depend on previous values
- We need to respect temporal order (no shuffling!)

### Components of Time Series

```
Time Series = Trend + Seasonality + Noise
```

1. **Trend:** Overall direction (up, down, flat)
2. **Seasonality:** Regular patterns that repeat (weekly, yearly)
3. **Noise:** Random fluctuations
4. **Cycles:** Longer patterns (economic cycles)

## Stationarity: The Key Concept 🔑

### What is Stationarity?
A time series is **stationary** if its statistical properties don't change over time:
- Mean is constant
- Variance is constant
- Autocovariance is constant

**Why does it matter?**
Most forecasting algorithms assume stationarity. Non-stationary data needs differencing.

### Visual Example
```
Stationary:          Non-Stationary:
  ┌─────┐              ┌───────────
  │  ~~│~              │     ~~~
  │~~  │               │  ~~~
Flat mean,           Trending upward,
constant variance    changing mean
```

### Making Data Stationary: Differencing
Take the difference between consecutive points:
```
Original: [10, 12, 15, 18, 22]
Differenced: [2, 3, 3, 4]
```

### Augmented Dickey-Fuller Test
Statistical test to check stationarity:
- p-value < 0.05 → Stationary ✓
- p-value ≥ 0.05 → Non-stationary (need differencing)

## Setting Up 🛠️

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# For time series
from statsmodels.tsa.stattools import adfuller, acf, pacf
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from sklearn.metrics import mean_squared_error, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 5)

print("Ready for time series forecasting! ⏰")

## Example 1: Load & Explore Time Series Data 📊

In [ ]:
# Create sample time series data (simulated daily stock prices)
np.random.seed(42)

# Generate dates
dates = pd.date_range(start='2020-01-01', periods=365, freq='D')

# Generate stock price with trend and noise
trend = np.linspace(100, 150, 365)
seasonal = 10 * np.sin(np.linspace(0, 4*np.pi, 365))
noise = np.random.normal(0, 3, 365)
price = trend + seasonal + noise

# Create DataFrame
df = pd.DataFrame({
    'date': dates,
    'price': price
})
df.set_index('date', inplace=True)

print("Time Series Data:")
print(df.head(10))
print(f"\nShape: {df.shape}")
print(f"Date range: {df.index.min()} to {df.index.max()}")

In [ ]:
# Visualize the time series
plt.figure(figsize=(14, 6))
plt.plot(df.index, df['price'], linewidth=2, label='Stock Price', color='#2E86AB')
plt.title('Time Series: Daily Stock Price (2020)', fontsize=14, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Price ($)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Statistics
print("Summary Statistics:")
print(df['price'].describe())

## Example 2: Test for Stationarity 🧪

In [ ]:
# Augmented Dickey-Fuller Test
def adf_test(timeseries, name):
    print(f"\n{'='*50}")
    print(f"ADF Test Results for: {name}")
    print(f"{'='*50}")
    
    result = adfuller(timeseries, autolag='AIC')
    
    print(f'ADF Statistic: {result[0]:.6f}')
    print(f'p-value: {result[1]:.6f}')
    print(f'Number of lags: {result[2]}')
    print(f'Number of observations: {result[3]}')
    
    print('\nCritical Values:')
    for key, value in result[4].items():
        print(f'  {key}: {value:.3f}')
    
    # Interpretation
    if result[1] <= 0.05:
        print(f"\n✓ Result: STATIONARY (p-value = {result[1]:.6f})")
        print("  Reject null hypothesis - series has no unit root")
    else:
        print(f"\n✗ Result: NON-STATIONARY (p-value = {result[1]:.6f})")
        print("  Fail to reject null hypothesis - needs differencing")

# Test original series
adf_test(df['price'], 'Original Price Series')

In [ ]:
# Difference the series
df['price_diff'] = df['price'].diff().dropna()

# Test differenced series
adf_test(df['price_diff'].dropna(), 'Differenced Price Series')

In [ ]:
# Visualize original vs differenced
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Original
axes[0].plot(df.index, df['price'], linewidth=2, color='#2E86AB')
axes[0].set_title('Original Price Series (Non-Stationary)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Price ($)')
axes[0].grid(True, alpha=0.3)

# Differenced
axes[1].plot(df.index[1:], df['price_diff'], linewidth=2, color='#A23B72')
axes[1].set_title('Differenced Price Series (Stationary)', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Price Change ($)')
axes[1].set_xlabel('Date')
axes[1].grid(True, alpha=0.3)
axes[1].axhline(y=0, color='red', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

## Example 3: Autocorrelation Analysis 📊

In [ ]:
# Calculate ACF and PACF
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ACF (AutoCorrelation Function)
acf_vals = acf(df['price_diff'].dropna(), nlags=40)
axes[0].bar(range(len(acf_vals)), acf_vals, color='#2E86AB', alpha=0.7)
axes[0].set_title('ACF - Autocorrelation Function', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Lag')
axes[0].set_ylabel('Autocorrelation')
axes[0].axhline(y=0, color='black', linestyle='-', linewidth=0.5)
axes[0].axhline(y=1.96/np.sqrt(len(df['price_diff'])), color='red', linestyle='--', label='95% CI')
axes[0].axhline(y=-1.96/np.sqrt(len(df['price_diff'])), color='red', linestyle='--')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# PACF (Partial AutoCorrelation Function)
pacf_vals = pacf(df['price_diff'].dropna(), nlags=40)
axes[1].bar(range(len(pacf_vals)), pacf_vals, color='#A23B72', alpha=0.7)
axes[1].set_title('PACF - Partial Autocorrelation Function', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Lag')
axes[1].set_ylabel('Partial Autocorrelation')
axes[1].axhline(y=0, color='black', linestyle='-', linewidth=0.5)
axes[1].axhline(y=1.96/np.sqrt(len(df['price_diff'])), color='red', linestyle='--', label='95% CI')
axes[1].axhline(y=-1.96/np.sqrt(len(df['price_diff'])), color='red', linestyle='--')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("ACF/PACF Interpretation Guide:")
print("- Bars outside red lines = significant correlation")
print("- Used to determine AR (p) and MA (q) parameters for ARIMA")

## Example 4: ARIMA Models 🎯

### What is ARIMA?

**ARIMA(p,d,q)** = AutoRegressive Integrated Moving Average

- **p (AR):** AutoRegressive - use past values
- **d (I):** Integrated - differencing order (how many times to difference)
- **q (MA):** Moving Average - use past errors

**Example:** ARIMA(1,1,1)
- Use 1 past value (AR)
- Difference once (I)
- Use 1 past error (MA)

In [ ]:
# Split data into train/test (respect temporal order!)
train_size = int(len(df) * 0.8)  # 80% for training
train = df['price'][:train_size]
test = df['price'][train_size:]

print(f"Training set: {len(train)} observations")
print(f"Test set: {len(test)} observations")
print(f"\nTrain period: {train.index[0]} to {train.index[-1]}")
print(f"Test period: {test.index[0]} to {test.index[-1]}")

In [ ]:
# Fit ARIMA model
# ARIMA(1,1,1) - simple baseline
model_arima = ARIMA(train, order=(1, 1, 1))
fitted_model = model_arima.fit()

print(fitted_model.summary())

In [ ]:
# Make predictions on test set
predictions_arima = fitted_model.forecast(steps=len(test))

# Calculate metrics
mae_arima = mean_absolute_error(test, predictions_arima)
rmse_arima = np.sqrt(mean_squared_error(test, predictions_arima))

print(f"ARIMA(1,1,1) Performance:")
print(f"  MAE (Mean Absolute Error): ${mae_arima:.2f}")
print(f"  RMSE (Root Mean Squared Error): ${rmse_arima:.2f}")

In [ ]:
# Visualize predictions
plt.figure(figsize=(14, 6))

# Plot actual data
plt.plot(train.index, train, label='Training Data', color='#2E86AB', linewidth=2)
plt.plot(test.index, test, label='Actual Test Data', color='#A23B72', linewidth=2)
plt.plot(test.index, predictions_arima, label='ARIMA(1,1,1) Predictions', 
         color='#F18F01', linestyle='--', linewidth=2)

# Add vertical line at train/test split
plt.axvline(x=train.index[-1], color='red', linestyle=':', alpha=0.5, label='Train/Test Split')

plt.title('ARIMA(1,1,1) Forecast vs Actual', fontsize=14, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Price ($)')
plt.legend(loc='best')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Example 5: Exponential Smoothing 🌟

### What is Exponential Smoothing?

Weights recent observations more heavily than older ones.

**Variants:**
- **Simple ES:** No trend/seasonality
- **Holt's ES:** With trend
- **Holt-Winters:** With trend + seasonality

In [ ]:
# Fit Exponential Smoothing (Holt-Winters)
# seasonal='add' for additive seasonality
model_es = ExponentialSmoothing(
    train,
    trend='add',
    seasonal='add',
    seasonal_periods=30  # Roughly monthly pattern
)
fitted_es = model_es.fit(optimized=True)

# Make predictions
predictions_es = fitted_es.forecast(steps=len(test))

# Calculate metrics
mae_es = mean_absolute_error(test, predictions_es)
rmse_es = np.sqrt(mean_squared_error(test, predictions_es))

print(f"Exponential Smoothing Performance:")
print(f"  MAE: ${mae_es:.2f}")
print(f"  RMSE: ${rmse_es:.2f}")

In [ ]:
# Compare both models
plt.figure(figsize=(14, 6))

plt.plot(train.index, train, label='Training Data', color='#2E86AB', linewidth=2)
plt.plot(test.index, test, label='Actual Test Data', color='#A23B72', linewidth=2)
plt.plot(test.index, predictions_arima, label=f'ARIMA(1,1,1) [RMSE: ${rmse_arima:.2f}]', 
         color='#F18F01', linestyle='--', linewidth=2)
plt.plot(test.index, predictions_es, label=f'Exp. Smoothing [RMSE: ${rmse_es:.2f}]', 
         color='#C73E1D', linestyle='-.', linewidth=2)

plt.axvline(x=train.index[-1], color='red', linestyle=':', alpha=0.5)
plt.title('Model Comparison: ARIMA vs Exponential Smoothing', fontsize=14, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Price ($)')
plt.legend(loc='best')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\n" + "="*50)
print("MODEL COMPARISON")
print("="*50)
print(f"ARIMA(1,1,1)         - RMSE: ${rmse_arima:.2f}, MAE: ${mae_arima:.2f}")
print(f"Exp. Smoothing       - RMSE: ${rmse_es:.2f}, MAE: ${mae_es:.2f}")
better = "ARIMA" if rmse_arima < rmse_es else "Exponential Smoothing"
print(f"\nBetter model: {better}")

## Example 6: Find Optimal ARIMA Parameters 🔍

### Grid Search Approach
Test different (p,d,q) combinations and find the best based on AIC

In [ ]:
# Grid search for best ARIMA parameters
# This can take a minute...

best_aic = float('inf')
best_params = None
results = []

print("Testing ARIMA parameters...")
print(f"{'p':<3} {'d':<3} {'q':<3} {'AIC':<10}")
print("-" * 25)

for p in range(0, 4):
    for d in range(0, 2):
        for q in range(0, 4):
            try:
                model = ARIMA(train, order=(p, d, q))
                fitted = model.fit()
                aic = fitted.aic
                results.append((p, d, q, aic))
                
                if aic < best_aic:
                    best_aic = aic
                    best_params = (p, d, q)
                
                if aic == best_aic:
                    print(f"{p:<3} {d:<3} {q:<3} {aic:<10.2f} ← Best")
            except:
                continue

print(f"\n✓ Best ARIMA parameters: {best_params}")
print(f"  AIC: {best_aic:.2f}")

In [ ]:
# Fit the best model
model_best = ARIMA(train, order=best_params)
fitted_best = model_best.fit()

predictions_best = fitted_best.forecast(steps=len(test))
mae_best = mean_absolute_error(test, predictions_best)
rmse_best = np.sqrt(mean_squared_error(test, predictions_best))

print(f"\nBest ARIMA{best_params} Performance:")
print(f"  MAE: ${mae_best:.2f}")
print(f"  RMSE: ${rmse_best:.2f}")
print(f"\nImprovement over ARIMA(1,1,1):")
print(f"  RMSE reduction: ${rmse_arima - rmse_best:.2f} ({((rmse_arima - rmse_best) / rmse_arima * 100):.1f}%)")

## Example 7: Forecasting Future Values 🔮

In [ ]:
# Refit on all data to get the best forecast
model_final = ARIMA(df['price'], order=best_params)
fitted_final = model_final.fit()

# Forecast 30 days into the future
forecast_steps = 30
future_forecast = fitted_final.forecast(steps=forecast_steps)

# Create future dates
last_date = df.index[-1]
future_dates = pd.date_range(start=last_date + pd.Timedelta(days=1), periods=forecast_steps, freq='D')

# Get confidence intervals
forecast_result = fitted_final.get_forecast(steps=forecast_steps)
forecast_ci = forecast_result.conf_int()

print(f"Forecast for next {forecast_steps} days:")
print(forecast_result.summary_table())

In [ ]:
# Visualize forecast with confidence intervals
plt.figure(figsize=(14, 6))

# Historical data
plt.plot(df.index[-100:], df['price'][-100:], label='Historical Data', color='#2E86AB', linewidth=2)

# Forecast
plt.plot(future_dates, future_forecast, label='30-Day Forecast', color='#F18F01', 
         linewidth=2, linestyle='--', marker='o', markersize=4)

# Confidence interval
plt.fill_between(future_dates,
                 forecast_ci.iloc[:, 0],
                 forecast_ci.iloc[:, 1],
                 color='#F18F01', alpha=0.2, label='95% Confidence Interval')

# Formatting
plt.axvline(x=df.index[-1], color='red', linestyle=':', alpha=0.5, label='Forecast Start')
plt.title(f'30-Day Price Forecast with Confidence Interval', fontsize=14, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Price ($)')
plt.legend(loc='best')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nForecast Summary:")
print(f"  Current Price: ${df['price'].iloc[-1]:.2f}")
print(f"  Forecasted 30-day avg: ${future_forecast.mean():.2f}")
print(f"  Forecasted range: ${future_forecast.min():.2f} - ${future_forecast.max():.2f}")

## Key Takeaways 📝

### Time Series Fundamentals
1. **Order matters** - can't shuffle time series data
2. **Stationarity is key** - test with Augmented Dickey-Fuller
3. **Differencing** - make data stationary
4. **Train/test split** - respect temporal order (don't shuffle)

### ARIMA(p,d,q)
- **p:** AR term (past values)
- **d:** Differencing order
- **q:** MA term (past errors)
- Use AIC to find optimal parameters

### When to Use What
| Situation | Best Approach |
|-----------|---------------|
| Simple trend, no seasonality | Exponential Smoothing |
| Complex patterns | ARIMA |
| Trend + strong seasonality | Holt-Winters |
| Very complex (images, text) | LSTM/Neural Networks |

### Evaluation Metrics
- **MAE:** Average absolute error (easy to interpret)
- **RMSE:** Penalizes large errors more
- **MAPE:** Percentage error (scale-independent)

### Best Practices
✓ Always check stationarity first  
✓ Use ACF/PACF to guide parameter selection  
✓ Grid search for optimal parameters  
✓ Compare multiple models  
✓ Validate on held-out test set  
✓ Monitor forecast uncertainty (confidence intervals)  

## Real-World Applications 🌍

### Stock Price Prediction
- Model daily/hourly returns
- Combine with technical indicators

### Weather Forecasting
- Temperature, precipitation, wind speed
- Often use ensemble methods

### Website Traffic
- Predict server load
- Plan infrastructure

### Sales Forecasting
- Inventory management
- Revenue planning

### COVID-19 Cases
- Epidemic forecasting
- Healthcare planning

### Energy Consumption
- Electricity demand
- Grid management

## Practice Challenge 💡

**Your Turn!** Try these exercises:

1. **Load Real Data:** Download weather data from a public source and build a temperature forecast
2. **Parameter Tuning:** Extend the grid search to larger ranges and find optimal parameters
3. **Ensemble Model:** Combine ARIMA and Exponential Smoothing predictions (average them)
4. **Multiple Series:** Forecast multiple related time series simultaneously
5. **Production Ready:** Save the model and create a pipeline for weekly retraining

**Resources:**
- [statsmodels ARIMA docs](https://www.statsmodels.org/stable/generated/statsmodels.tsa.arima.model.ARIMA.html)
- [Kaggle Datasets](https://www.kaggle.com/) - many time series datasets
- [Forecasting: Principles and Practice](https://otexts.com/fpp2/) - free textbook